In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
master_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/default/week7_data/customer_master.csv")

display(master_df)

customer_id,customer_name,city,email,phone,status
C001,Manish Khyalia,Sikar,manish.khyalia@gmail.com,9876543201,Active
C002,Rahul Meel,Jaipur,rahul.meel@gmail.com,9876543202,Active
C003,Pooja Bhukar,Bikaner,pooja.bhukar@gmail.com,9876543203,Inactive
C004,Amit Bhaskar,Jodhpur,amit.bhaskar@gmail.com,9876543204,Active
C005,Neha Sharma,Delhi,neha.sharma@gmail.com,9876543205,Active
C006,Rohit Verma,Ajmer,null,9876543206,Inactive
C007,Kavita Singh,Udaipur,kavita.singh@gmail.com,9876543207,Active
C008,Vivek Patel,Kota,vivek.patel@gmail.com,9876543208,Active
C009,Anjali Jain,Ahmedabad,anjali.jain@gmail.com,9876543209,Inactive
C010,Sachin Gupta,Pune,sachin.gupta@gmail.com,9876543210,Active


In [0]:
incremental_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/default/week7_data/customer_incremental.csv")

display(incremental_df)

customer_id,customer_name,city,email,phone,status
C003,Pooja Bhukar,Jaipur,pooja.bhukar@company.com,9876543203,Active
C007,Kavita Singh,Udaipur,kavita.singh@company.com,9876543207,Inactive
C010,Sachin Gupta,Pune,sachin.gupta@company.com,9999999999,Active
C015,Komal Yadav,Lucknow,komal.yadav@gmail.com,9876543215,Active
C018,Harsh Godara,Sikar,harsh.godara@company.com,9999998888,Active
C021,Rajveer Jakhar,Sikar,rajveer.jakhar@gmail.com,9876543221,Active
C022,Ankit Sheoran,Rohtak,ankit.sheoran@gmail.com,9876543222,Active
C023,Priyanshu Dhaka,Jaipur,priyanshu.dhaka@gmail.com,9876543223,Inactive
C024,Megha Poonia,Bikaner,megha.poonia@gmail.com,9876543224,Active
C025,Nitin Chahar,Kota,nitin.chahar@gmail.com,9876543225,Active


In [0]:
master_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer_master_delta")

In [0]:
display(spark.table("customer_master_delta"))

customer_id,customer_name,city,email,phone,status
C001,Manish Khyalia,Sikar,manish.khyalia@gmail.com,9876543201,Active
C002,Rahul Meel,Jaipur,rahul.meel@gmail.com,9876543202,Active
C003,Pooja Bhukar,Bikaner,pooja.bhukar@gmail.com,9876543203,Inactive
C004,Amit Bhaskar,Jodhpur,amit.bhaskar@gmail.com,9876543204,Active
C005,Neha Sharma,Delhi,neha.sharma@gmail.com,9876543205,Active
C006,Rohit Verma,Ajmer,null,9876543206,Inactive
C007,Kavita Singh,Udaipur,kavita.singh@gmail.com,9876543207,Active
C008,Vivek Patel,Kota,vivek.patel@gmail.com,9876543208,Active
C009,Anjali Jain,Ahmedabad,anjali.jain@gmail.com,9876543209,Inactive
C010,Sachin Gupta,Pune,sachin.gupta@gmail.com,9876543210,Active


In [0]:
clean_df = spark.table("customer_master_delta") \
    .dropDuplicates(["customer_id"])

In [0]:
clean_df = clean_df.fillna({
    "email": "Not Available"
})

In [0]:
display(clean_df)

customer_id,customer_name,city,email,phone,status
C001,Manish Khyalia,Sikar,manish.khyalia@gmail.com,9876543201,Active
C002,Rahul Meel,Jaipur,rahul.meel@gmail.com,9876543202,Active
C003,Pooja Bhukar,Bikaner,pooja.bhukar@gmail.com,9876543203,Inactive
C004,Amit Bhaskar,Jodhpur,amit.bhaskar@gmail.com,9876543204,Active
C005,Neha Sharma,Delhi,neha.sharma@gmail.com,9876543205,Active
C006,Rohit Verma,Ajmer,Not Available,9876543206,Inactive
C007,Kavita Singh,Udaipur,kavita.singh@gmail.com,9876543207,Active
C008,Vivek Patel,Kota,vivek.patel@gmail.com,9876543208,Active
C009,Anjali Jain,Ahmedabad,anjali.jain@gmail.com,9876543209,Inactive
C010,Sachin Gupta,Pune,sachin.gupta@gmail.com,9876543210,Active


In [0]:
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer_master_delta")

In [0]:
delta_table = DeltaTable.forName(
    spark,
    "customer_master_delta"
)

In [0]:
(
    delta_table.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(set={
        "customer_name": "source.customer_name",
        "city": "source.city",
        "email": "source.email",
        "phone": "source.phone",
        "status": "source.status"
    })
    .whenNotMatchedInsert(values={
        "customer_id": "source.customer_id",
        "customer_name": "source.customer_name",
        "city": "source.city",
        "email": "source.email",
        "phone": "source.phone",
        "status": "source.status"
    })
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
    spark.table("customer_master_delta")
)

customer_id,customer_name,city,email,phone,status
C005,Neha Sharma,Delhi,neha.sharma@gmail.com,9876543205,Active
C009,Anjali Jain,Ahmedabad,anjali.jain@gmail.com,9876543209,Inactive
C004,Amit Bhaskar,Jodhpur,amit.bhaskar@gmail.com,9876543204,Active
C012,Deepak Rathore,Indore,deepak.rathore@gmail.com,9876543212,Inactive
C013,Nisha Joshi,Hyderabad,nisha.joshi@gmail.com,9876543213,Active
C008,Vivek Patel,Kota,vivek.patel@gmail.com,9876543208,Active
C017,Sneha Mehta,Mumbai,sneha.mehta@gmail.com,9876543217,Active
C020,Mohit Beniwal,Hisar,mohit.beniwal@gmail.com,9876543220,Active
C001,Manish Khyalia,Sikar,manish.khyalia@gmail.com,9876543201,Active
C014,Arjun Saini,Chandigarh,arjun.saini@gmail.com,9876543214,Active


In [0]:
spark.table("customer_master_delta") \
.groupBy("customer_id") \
.count() \
.filter("count > 1") \
.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [0]:
display(
    spark.table("customer_master_delta")
)

customer_id,customer_name,city,email,phone,status
C005,Neha Sharma,Delhi,neha.sharma@gmail.com,9876543205,Active
C009,Anjali Jain,Ahmedabad,anjali.jain@gmail.com,9876543209,Inactive
C004,Amit Bhaskar,Jodhpur,amit.bhaskar@gmail.com,9876543204,Active
C012,Deepak Rathore,Indore,deepak.rathore@gmail.com,9876543212,Inactive
C013,Nisha Joshi,Hyderabad,nisha.joshi@gmail.com,9876543213,Active
C008,Vivek Patel,Kota,vivek.patel@gmail.com,9876543208,Active
C017,Sneha Mehta,Mumbai,sneha.mehta@gmail.com,9876543217,Active
C020,Mohit Beniwal,Hisar,mohit.beniwal@gmail.com,9876543220,Active
C001,Manish Khyalia,Sikar,manish.khyalia@gmail.com,9876543201,Active
C014,Arjun Saini,Chandigarh,arjun.saini@gmail.com,9876543214,Active
